# 🎓 **Universidad Pedagógica y Tecnológica de Colombia (UPTC)**
## **Especialización en Inteligencia Artificial**
### **Módulo: Ciencia de Datos y Toma decisiones**
---
### **Taller Práctico Integrado: Analítica Académica con SQL, Extracción de Datos y Flujos Text-to-SQL**
* **Estudiante:** José Daniel Rojas Verbel
* **Código:** 202624475
* **Base de datos:** university.db (SQLite relacional)
* **Herramientas:** Python 3, SQLite3, Pandas, Google Antigravity / Google Colab
---


## 1. Contexto y Objetivos del Taller

La Universidad ha decidido fortalecer sus procesos de analítica institucional mediante un sistema de inteligencia académica conectado al módulo de registro. Los directivos requieren responder preguntas estratégicas sobre rendimiento estudiantil, ocupación de cursos, distribución docente, comportamiento de matrículas y uso de infraestructura física.

En proyectos de Inteligencia Artificial, los datos operacionales residen habitualmente en bases de datos relacionales normalizadas. El objetivo de este taller es dominar la extracción, manipulación, cruce y agregación de información mediante SQL sobre la base de datos académica (university.db), integrando tanto el análisis manual como la auditoría crítica de agentes de IA (**Text-to-SQL**) dentro del entorno Google Antigravity.

---


## 2. Configuración del Entorno de Ejecución (Google Colab / Local)

En este bloque preparamos la conexión a la base de datos university.db utilizando sqlite3 y pandas. Si estás ejecutando en Google Colab, puedes clonar el repositorio o subir el archivo de base de datos directamente.


In [2]:
# 1. Importación de librerías para manejo de datos y conexión SQL
import sqlite3
import pandas as pd
import os
import urllib.request

# Configuración visual de Pandas para visualizar todas las columnas con claridad
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.colheader_justify', 'left')

# Comprobar si estamos en Google Colab o en Antigravity local
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# URL del archivo binario en tu GitHub (raw)
url_github = "https://raw.githubusercontent.com/joserojas07-bit/IA-CienciaDatos2026-2/main/TAREA_3_SQL/DATOS/university.db"
db_local = "university.db"

# Descargar solo si no existe en el entorno
if not os.path.exists(db_local):
    print("⬇️ Descargando university.db desde GitHub...")
    urllib.request.urlretrieve(url_github, db_local)
    print("✅ Archivo descargado exitosamente en Colab.")
else:
    print("ℹ️ El archivo ya existe en el entorno de Colab.")

# Conectar
conn = sqlite3.connect(db_local)
print(f"🔗 Conexión SQLite lista sobre: {db_local}")

# Función para ejecutar consultas
def run_query(query, description=""):
    if description:
        print(f"📌 {description}")
    return pd.read_sql_query(query, conn)

# Función auxiliar para ejecutar consultas y retornar DataFrame formateado
def run_query(query, description=""):
    if description:
        print(f"📌 {description}")
    return pd.read_sql_query(query, conn)

⬇️ Descargando university.db desde GitHub...
✅ Archivo descargado exitosamente en Colab.
🔗 Conexión SQLite lista sobre: university.db


## 3. Inspección del Esquema y Modelo Relacional

La base de datos cuenta con **11 tablas principales** que representan la gestión académica:
1. FACULTADES
2. CARRERAS
3. CURSOS
4. ROLES (1 = Estudiante, 2 = Profesor)
5. USUARIOS
6. EDIFICIOS
7. SALONES
8. CALENDARIO_CURSOS
9. CARRERAS_ESTUDIANTES
10. CURSOS_CARRERAS
11. CURSOS_ESTUDIANTES
"@

Add-Code @"
# Listar todas las tablas existentes en university.db y su conteo de registros
query_tables = '''
SELECT name AS Tabla 
FROM sqlite_master 
WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
ORDER BY name;
'''
tablas_df = run_query(query_tables, "Tablas de la base de datos universitaria:")
conteos = []
for tabla in tablas_df['Tabla']:
    c = run_query(f"SELECT COUNT(*) as Conteo FROM {tabla}").iloc[0]['Conteo']
    conteos.append(c)

tablas_df['Total_Registros'] = conteos
tablas_df


---
# PARTE I: BATERÍA PROGRESIVA DE CONSULTAS RELACIONALES

---
## Bloque A: Exploración, Ordenamiento y Filtros Básicos

Consultas de selección, ordenamiento y filtros condicionales para auditoría preliminar de usuarios, carreras y catálogo de asignaturas.


### Consulta A.1: Mostrar todos los registros de la tabla USUARIOS
Permite verificar la totalidad del censo universitario cargado en el sistema (estudiantes y docentes).


In [3]:
q_a1 = '''
SELECT * 
FROM USUARIOS;
'''
run_query(q_a1, "A.1. Todos los registros de la tabla USUARIOS")


📌 A.1. Todos los registros de la tabla USUARIOS


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
0,1,Ben,Stone,0,2
1,2,Grace,Stone,0,2
2,3,Mikaela,Stone,0,2
3,4,Jared,Vásquez,0,2
4,5,Zeke,Landon,0,2
5,6,Saanvi,Bahl,0,2
6,7,Ulrich,Nielsen,0,2
7,8,Charlotte,Doppler,0,2
8,9,Claudia,Tiedemann,0,2
9,10,Robert,Vance,0,2


### Consulta A.2: Mostrar únicamente el NOMBRE y APELLIDO de todos los usuarios
Proyección acotada para listados de asistencia o directorios sin exponer datos de rol o semestre.


In [4]:
q_a2 = '''
SELECT NOMBRE, APELLIDO 
FROM USUARIOS;
'''
run_query(q_a2, "A.2. Directorio básico (Nombre y Apellido)")


📌 A.2. Directorio básico (Nombre y Apellido)


,NOMBRE,APELLIDO
0,Ben,Stone
1,Grace,Stone
2,Mikaela,Stone
3,Jared,Vásquez
4,Zeke,Landon
5,Saanvi,Bahl
6,Ulrich,Nielsen
7,Charlotte,Doppler
8,Claudia,Tiedemann
9,Robert,Vance


### Consulta A.3: Mostrar todas las carreras ordenadas alfabéticamente de forma ascendente
Catálogo ordenado alfabéticamente (ASC) para navegación institucional y menús de selección.


In [5]:
q_a3 = '''
SELECT ID_CARRERA, NOMBRE, ID_FACULTAD 
FROM CARRERAS 
ORDER BY NOMBRE ASC;
'''
run_query(q_a3, "A.3. Carreras ordenadas alfabéticamente (A-Z)")


📌 A.3. Carreras ordenadas alfabéticamente (A-Z)


,ID_CARRERA,NOMBRE,ID_FACULTAD
0,4,Filosofía,2
1,2,Ingeniería Mecánica,1
2,1,Ingeniería de Sistemas,1
3,3,Matemáticas Aplicadas,1


### Consulta A.4: Mostrar los cursos ordenados alfabéticamente de forma descendente
Inspección inversa (DESC) del catálogo global de asignaturas.


In [6]:
q_a4 = '''
SELECT ID_CURSO, NOMBRE, ID_FACULTAD 
FROM CURSOS 
ORDER BY NOMBRE DESC;
'''
run_query(q_a4, "A.4. Cursos ordenados alfabéticamente (Z-A)")


📌 A.4. Cursos ordenados alfabéticamente (Z-A)


,ID_CURSO,NOMBRE,ID_FACULTAD
0,67,Ética,2.0
1,9,Árboles y Grafos,1.0
2,49,Álgebra Moderna,1.0
3,45,Álgebra Lineal,1.0
4,4,Técnicas y Prácticas de Programación,1.0
...,...,...,...
66,10,Arquitectura del Computador,1.0
67,20,Aprendizaje Automático,1.0
68,16,Análisis y Diseño de Algoritmos,1.0
69,65,Antropología Filosófica,2.0


### Consulta A.5: Mostrar los estudiantes que estén cursando el semestre 2
Filtrado por rol (ID_ROL = 1 Estudiante) y condición de semestre activo (SEMESTRE = 2).


In [7]:
q_a5 = '''
SELECT ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE 
FROM USUARIOS 
WHERE ID_ROL = 1 
  AND SEMESTRE = 2;
'''
run_query(q_a5, "A.5. Estudiantes en semestre 2")


📌 A.5. Estudiantes en semestre 2


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
0,20,Mads,Nielsen,2


### Consulta A.6: Mostrar los usuarios cuyo apellido sea 'Gomez'
Búsqueda nominal exacta sobre el campo APELLIDO.


In [8]:
q_a6 = '''
SELECT ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE, ID_ROL 
FROM USUARIOS 
WHERE APELLIDO = 'Gomez';
'''
run_query(q_a6, "A.6. Usuarios con apellido Gomez")


📌 A.6. Usuarios con apellido Gomez


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL


### Consulta A.7: Mostrar los estudiantes que pertenezcan al semestre 1 o al semestre 2
Segmentación de cohorte inicial (ciclo básico universitario) mediante cláusula IN (1, 2).


In [9]:
q_a7 = '''
SELECT ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE 
FROM USUARIOS 
WHERE ID_ROL = 1 
  AND SEMESTRE IN (1, 2);
'''
run_query(q_a7, "A.7. Estudiantes de semestres 1 y 2")


📌 A.7. Estudiantes de semestres 1 y 2


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE
0,12,Cal,Stone,1
1,18,Elisabeth,Doppler,1
2,20,Mads,Nielsen,2


### Consulta A.8: Asignaturas de Ingeniería y Ciencias relacionadas con Inteligencia o Datos
Filtro de facultad (ID_FACULTAD = 1) con patrones de coincidencia parcial de texto (LIKE con %Inteligentes%, %Aprendizaje%, %Datos%).


In [10]:
q_a8 = '''
SELECT ID_CURSO, NOMBRE 
FROM CURSOS 
WHERE ID_FACULTAD = 1 
  AND (
      NOMBRE LIKE '%Inteligentes%' 
      OR NOMBRE LIKE '%Aprendizaje%' 
      OR NOMBRE LIKE '%Datos%'
  )
ORDER BY NOMBRE ASC;
'''
run_query(q_a8, "A.8. Cursos de IA y Ciencia de Datos en Facultad de Ingeniería")


📌 A.8. Cursos de IA y Ciencia de Datos en Facultad de Ingeniería


,ID_CURSO,NOMBRE
0,20,Aprendizaje Automático
1,14,Comunicación de Datos
2,3,Estructuras de Datos
3,17,Gestión y Modelación de Datos
4,24,Sistemas Inteligentes


---
## Bloque B: Manipulación de Datos — DML (INSERT, UPDATE, DELETE)

Validación transaccional para altas, modificaciones y bajas de registros con verificación mediante SELECT.


### Consultas B.1 a B.3: Inserción, Eliminación y Verificación de una Nueva Facultad


In [14]:
cursor = conn.cursor()

# B.1: Inserción de nueva facultad
cursor.execute('''
INSERT INTO FACULTADES (ID_FACULTAD, NOMBRE) 
VALUES (99, 'Facultad de Inteligencia Artificial y Robótica');
''')
conn.commit()
print("B.1: Facultad 99 insertada con éxito.")

# Comprobación de inserción
df_inserted = pd.read_sql_query("SELECT * FROM FACULTADES WHERE ID_FACULTAD = 99;", conn)
print("Registro actual:", df_inserted.to_dict(orient='records'))

# B.2: Eliminación de la facultad recién insertada
cursor.execute("DELETE FROM FACULTADES WHERE ID_FACULTAD = 99;")
conn.commit()
print("B.2: Facultad 99 eliminada.")

# B.3: Verificación de eliminación
q_b3 = "SELECT * FROM FACULTADES WHERE ID_FACULTAD = 99;"
run_query(q_b3, "B.3. Verificación de eliminación (debe retornar DataFrame vacío)")


B.1: Facultad 99 insertada con éxito.
Registro actual: [{'ID_FACULTAD': 99, 'NOMBRE': 'Facultad de Inteligencia Artificial y Robótica'}]
B.2: Facultad 99 eliminada.
📌 B.3. Verificación de eliminación (debe retornar DataFrame vacío)


,ID_FACULTAD,NOMBRE


### Consultas B.4 a B.8: Inserción, Actualización y Eliminación de un Nuevo Estudiante


In [16]:
# B.4: Insertar nuevo estudiante en USUARIOS (ID_ROL = 1)
cursor.execute('''
INSERT INTO USUARIOS (ID_USUARIO, NOMBRE, APELLIDO, SEMESTRE, ID_ROL) 
VALUES (22, 'Daniel', 'Rojas', 1, 1);
''')
conn.commit()

# B.5: Verificar que se haya guardado correctamente
print("B.5. Verificación de inserción del estudiante:")
display(run_query("SELECT * FROM USUARIOS WHERE ID_USUARIO = 22;"))

# B.6: Actualizar el semestre del estudiante con ID_USUARIO = 22 a semestre 4
cursor.execute("UPDATE USUARIOS SET SEMESTRE = 4 WHERE ID_USUARIO = 22;")
conn.commit()
print("B.6. Verificación de actualización a semestre 4:")
display(run_query("SELECT * FROM USUARIOS WHERE ID_USUARIO = 22;"))

# B.7: Eliminar el registro del estudiante
cursor.execute("DELETE FROM USUARIOS WHERE ID_USUARIO = 22;")
conn.commit()

# B.8: Verificar que el registro haya sido eliminado
print("B.8. Verificación final de eliminación:")
display(run_query("SELECT * FROM USUARIOS WHERE ID_USUARIO = 22;"))


B.5. Verificación de inserción del estudiante:


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
0,22,Daniel,Rojas,1,1


B.6. Verificación de actualización a semestre 4:


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL
0,22,Daniel,Rojas,4,1


B.8. Verificación final de eliminación:


,ID_USUARIO,NOMBRE,APELLIDO,SEMESTRE,ID_ROL


---
## Bloque C: Cruces Relacionales y Carteleras (INNER JOIN)

Extracción cruzada para vincular entidades: carreras con facultades, roles de usuarios, asignaturas matriculadas, programación de aulas y carteleras horarias.


### Consulta C.1: Mostrar cada carrera junto con el nombre de su facultad


In [17]:
q_c1 = '''
SELECT 
    c.ID_CARRERA,
    c.NOMBRE AS CARRERA,
    f.NOMBRE AS FACULTAD
FROM CARRERAS c
INNER JOIN FACULTADES f ON c.ID_FACULTAD = f.ID_FACULTAD
ORDER BY f.NOMBRE, c.NOMBRE;
'''
run_query(q_c1, "C.1. Carreras con sus respectivas Facultades")


📌 C.1. Carreras con sus respectivas Facultades


,ID_CARRERA,CARRERA,FACULTAD
0,4,Filosofía,Humanidades y Ciencias Sociales
1,2,Ingeniería Mecánica,Ingeniería y Ciencias
2,1,Ingeniería de Sistemas,Ingeniería y Ciencias
3,3,Matemáticas Aplicadas,Ingeniería y Ciencias


### Consulta C.2: Mostrar el nombre de solo los usuarios que tienen rol de estudiante


In [18]:
q_c2 = '''
SELECT 
    u.ID_USUARIO,
    u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
    u.SEMESTRE,
    r.NOMBRE AS ROL
FROM USUARIOS u
INNER JOIN ROLES r ON u.ID_ROL = r.ID_ROL
WHERE r.NOMBRE = 'Estudiante'
ORDER BY u.SEMESTRE, ESTUDIANTE;
'''
run_query(q_c2, "C.2. Usuarios con rol de Estudiante")


📌 C.2. Usuarios con rol de Estudiante


,ID_USUARIO,ESTUDIANTE,SEMESTRE,ROL
0,12,Cal Stone,1,Estudiante
1,18,Elisabeth Doppler,1,Estudiante
2,20,Mads Nielsen,2,Estudiante
3,13,Angelina Meyer,5,Estudiante
4,16,Bartosz Tiedemann,5,Estudiante
5,17,Franziska Doppler,5,Estudiante
6,14,Jonas Kahnwald,5,Estudiante
7,15,Martha Nielsen,5,Estudiante
8,11,Olive Stone,5,Estudiante
9,19,Magnus Nielsen,6,Estudiante


### Consulta C.3: Mostrar el nombre de solo los usuarios que tienen rol de profesor


In [19]:
q_c3 = '''
SELECT 
    u.ID_USUARIO,
    u.NOMBRE || ' ' || u.APELLIDO AS PROFESOR,
    r.NOMBRE AS ROL
FROM USUARIOS u
INNER JOIN ROLES r ON u.ID_ROL = r.ID_ROL
WHERE r.NOMBRE = 'Profesor'
ORDER BY PROFESOR;
'''
run_query(q_c3, "C.3. Usuarios con rol de Profesor")


📌 C.3. Usuarios con rol de Profesor


,ID_USUARIO,PROFESOR,ROL
0,1,Ben Stone,Profesor
1,8,Charlotte Doppler,Profesor
2,9,Claudia Tiedemann,Profesor
3,2,Grace Stone,Profesor
4,4,Jared Vásquez,Profesor
5,3,Mikaela Stone,Profesor
6,10,Robert Vance,Profesor
7,6,Saanvi Bahl,Profesor
8,7,Ulrich Nielsen,Profesor
9,5,Zeke Landon,Profesor


### Consulta C.4: Cursos matriculados por cada estudiante
Cruce de 4 tablas: USUARIOS -> CURSOS_ESTUDIANTES -> CALENDARIO_CURSOS -> CURSOS.


In [20]:
q_c4 = '''
SELECT 
    u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
    cur.NOMBRE AS CURSO
FROM CURSOS_ESTUDIANTES ce
INNER JOIN USUARIOS u ON ce.ID_USUARIO = u.ID_USUARIO
INNER JOIN CALENDARIO_CURSOS cc ON ce.ID_CALENDARIO = cc.ID_CALENDARIO
INNER JOIN CURSOS cur ON cc.ID_CURSO = cur.ID_CURSO
ORDER BY ESTUDIANTE, CURSO;
'''
df_c4 = run_query(q_c4, "C.4. Asignaturas inscritas por cada estudiante")
print(f"Total inscripciones registradas: {len(df_c4)}")
df_c4.head(15)


📌 C.4. Asignaturas inscritas por cada estudiante
Total inscripciones registradas: 151


,ESTUDIANTE,CURSO
0,Angelina Meyer,Filosofía Antigua
1,Angelina Meyer,Teología
2,Bartosz Tiedemann,Cinemática y Dinámica
3,Bartosz Tiedemann,Cinemática y Dinámica
4,Bartosz Tiedemann,Cinemática y Dinámica
5,Bartosz Tiedemann,Cálculo Diferencial
6,Bartosz Tiedemann,Cálculo Diferencial
7,Bartosz Tiedemann,Cálculo Integral
8,Bartosz Tiedemann,Cálculo Integral
9,Bartosz Tiedemann,Cálculo Multivariado


### Consulta C.5: Cursos programados con salón y edificio donde se dictan
Relaciona la programación académica con la infraestructura física institucional (SALONES y EDIFICIOS).


In [21]:
q_c5 = '''
SELECT 
    cc.ID_CALENDARIO,
    cur.NOMBRE AS CURSO,
    s.NOMBRE AS SALON,
    e.NOMBRE AS EDIFICIO,
    cc.DIA,
    cc.HORA_INICIO || ' - ' || cc.HORA_FIN AS HORARIO
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS cur ON cc.ID_CURSO = cur.ID_CURSO
INNER JOIN SALONES s ON cc.ID_SALON = s.ID_SALON
INNER JOIN EDIFICIOS e ON s.ID_EDIFICIO = e.ID_EDIFICIO
ORDER BY cur.NOMBRE, cc.DIA;
'''
df_c5 = run_query(q_c5, "C.5. Programación de cursos con salón y edificio")
df_c5.head(10)


📌 C.5. Programación de cursos con salón y edificio


,ID_CALENDARIO,CURSO,SALON,EDIFICIO,DIA,HORARIO
0,33,Antropología Filosófica,3.6,Almendros,Martes,08:00 AM - 10:00 AM
1,68,Aprendizaje Automático,1.1,El Samán,Jueves,04:00 PM - 06:00 PM
2,67,Aprendizaje Automático,1.1,El Samán,Martes,04:00 PM - 06:00 PM
3,11,Arquitectura del Computador,2.6,El Lago,Lunes,04:00 PM - 06:00 PM
4,12,Arquitectura del Computador,2.6,El Lago,Viernes,04:00 PM - 06:00 PM
5,35,Cinemática y Dinámica,2.2,El Lago,Lunes,02:00 PM - 04:00 PM
6,36,Cinemática y Dinámica,2.2,El Lago,Miércoles,02:00 PM - 04:00 PM
7,37,Cinemática y Dinámica,2.2,El Lago,Viernes,02:00 PM - 04:00 PM
8,14,Cálculo Diferencial,2.5,El Lago,Jueves,08:00 AM - 10:00 AM
9,13,Cálculo Diferencial,2.5,El Lago,Martes,08:00 AM - 10:00 AM


### Consulta C.6: Cartelera de programación para el día Martes
Incluye curso, horario, salón, edificio y nombre completo del docente a cargo.


In [22]:
q_c6 = '''
SELECT 
    cur.NOMBRE AS CURSO,
    cc.HORA_INICIO,
    cc.HORA_FIN,
    s.NOMBRE AS SALON,
    e.NOMBRE AS EDIFICIO,
    prof.NOMBRE || ' ' || prof.APELLIDO AS PROFESOR
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS cur ON cc.ID_CURSO = cur.ID_CURSO
INNER JOIN SALONES s ON cc.ID_SALON = s.ID_SALON
INNER JOIN EDIFICIOS e ON s.ID_EDIFICIO = e.ID_EDIFICIO
INNER JOIN USUARIOS prof ON cc.ID_PROFESOR = prof.ID_USUARIO
WHERE cc.DIA = 'Martes'
ORDER BY cc.HORA_INICIO, cur.NOMBRE;
'''
run_query(q_c6, "C.6. Cartelera académica del día Martes")


📌 C.6. Cartelera académica del día Martes


,CURSO,HORA_INICIO,HORA_FIN,SALON,EDIFICIO,PROFESOR
0,Cálculo Multivariado,02:00 PM,04:00 PM,2.1,El Lago,Jared Vásquez
1,Desarrollo de Videouegos,02:00 PM,05:00 PM,3.2,Palmas,Ben Stone
2,Propiedades de los Materiales,02:00 PM,04:00 PM,3.5,El Lago,Claudia Tiedemann
3,Transferencia de Calor,02:00 PM,04:00PM,1.5,El Samán,Saanvi Bahl
4,Aprendizaje Automático,04:00 PM,06:00 PM,1.1,El Samán,Robert Vance
5,Diseño Mecánico,04:00 PM,06:00 PM,3.4,El Lago,Claudia Tiedemann
6,Máquinas Térmicas e Hidráulicas,04:00 PM,06:00 PM,1.5,El Samán,Saanvi Bahl
7,Antropología Filosófica,08:00 AM,10:00 AM,3.6,Almendros,Zeke Landon
8,Cálculo Diferencial,08:00 AM,10:00 AM,2.5,El Lago,Grace Stone
9,Introducción a la Programación,08:00 AM,10:00 AM,1.4,El Lago,Ben Stone


### Consulta C.7: Horario completo de un estudiante específico (ID = 1)
Ordenado cronológicamente por día de la semana y hora de inicio.


In [23]:
q_c7 = '''
SELECT 
    u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
    cc.DIA,
    cc.HORA_INICIO,
    cc.HORA_FIN,
    cur.NOMBRE AS CURSO,
    s.NOMBRE AS SALON,
    e.NOMBRE AS EDIFICIO
FROM CURSOS_ESTUDIANTES ce
INNER JOIN USUARIOS u ON ce.ID_USUARIO = u.ID_USUARIO
INNER JOIN CALENDARIO_CURSOS cc ON ce.ID_CALENDARIO = cc.ID_CALENDARIO
INNER JOIN CURSOS cur ON cc.ID_CURSO = cur.ID_CURSO
INNER JOIN SALONES s ON cc.ID_SALON = s.ID_SALON
INNER JOIN EDIFICIOS e ON s.ID_EDIFICIO = e.ID_EDIFICIO
WHERE u.ID_USUARIO = 1
ORDER BY 
    CASE cc.DIA
        WHEN 'Lunes' THEN 1
        WHEN 'Martes' THEN 2
        WHEN 'Miercoles' THEN 3
        WHEN 'Jueves' THEN 4
        WHEN 'Viernes' THEN 5
        WHEN 'Sabado' THEN 6
        ELSE 7
    END,
    cc.HORA_INICIO;
'''
run_query(q_c7, "C.7. Horario semanal consolidado del Estudiante ID = 1")


📌 C.7. Horario semanal consolidado del Estudiante ID = 1


,ESTUDIANTE,DIA,HORA_INICIO,HORA_FIN,CURSO,SALON,EDIFICIO


### Consultas C.8 y C.9: Profesores y Cursos Ofertados que Orientan


In [24]:
# C.8: Profesores que dictan cursos ofertados vs profesores registrados
q_c8 = '''
SELECT DISTINCT 
    u.ID_USUARIO,
    u.NOMBRE || ' ' || u.APELLIDO AS PROFESOR,
    'Con cursos ofertados en calendario' AS ESTADO
FROM USUARIOS u
INNER JOIN CALENDARIO_CURSOS cc ON u.ID_USUARIO = cc.ID_PROFESOR
WHERE u.ID_ROL = 2

UNION

SELECT 
    u.ID_USUARIO,
    u.NOMBRE || ' ' || u.APELLIDO AS PROFESOR,
    'Docente de planta registrado' AS ESTADO
FROM USUARIOS u
WHERE u.ID_ROL = 2
ORDER BY PROFESOR;
'''
display(run_query(q_c8, "C.8. Auditoría docente: Estado de asignación en calendario"))

# C.9: Profesores con los nombres de cursos ofertados que orientan
q_c9 = '''
SELECT DISTINCT
    prof.NOMBRE || ' ' || prof.APELLIDO AS PROFESOR,
    cur.NOMBRE AS CURSO
FROM CALENDARIO_CURSOS cc
INNER JOIN USUARIOS prof ON cc.ID_PROFESOR = prof.ID_USUARIO
INNER JOIN CURSOS cur ON cc.ID_CURSO = cur.ID_CURSO
ORDER BY PROFESOR, CURSO;
'''
display(run_query(q_c9, "C.9. Asignación académica de cursos por docente"))


📌 C.8. Auditoría docente: Estado de asignación en calendario


,ID_USUARIO,PROFESOR,ESTADO
0,1,Ben Stone,Con cursos ofertados en calendario
1,1,Ben Stone,Docente de planta registrado
2,8,Charlotte Doppler,Con cursos ofertados en calendario
3,8,Charlotte Doppler,Docente de planta registrado
4,9,Claudia Tiedemann,Con cursos ofertados en calendario
5,9,Claudia Tiedemann,Docente de planta registrado
6,2,Grace Stone,Con cursos ofertados en calendario
7,2,Grace Stone,Docente de planta registrado
8,4,Jared Vásquez,Con cursos ofertados en calendario
9,4,Jared Vásquez,Docente de planta registrado


📌 C.9. Asignación académica de cursos por docente


,PROFESOR,CURSO
0,Ben Stone,Desarrollo de Videouegos
1,Ben Stone,Estructuras de Datos
2,Ben Stone,Introducción a la Programación
3,Ben Stone,Técnicas y Prácticas de Programación
4,Charlotte Doppler,Dibujo Mecánico
5,Claudia Tiedemann,Diseño Mecánico
6,Claudia Tiedemann,Máquinas Térmicas e Hidráulicas
7,Claudia Tiedemann,Propiedades de los Materiales
8,Claudia Tiedemann,Química y Ciencia de Materiales
9,Claudia Tiedemann,Álgebra Moderna


---
## Bloque D: Auditoría de Registros Huérfanos (LEFT JOIN / Subconsultas)

Identificación de inconsistencias operacionales, estudiantes inactivos o cursos desiertos sin matrícula mediante LEFT JOIN ... WHERE ... IS NULL.


### Consulta D.1: Estudiantes registrados sin ningún curso matriculado
Auditoría para detectar deserción o alumnos no matriculados en el periodo.


In [25]:
q_d1 = '''
SELECT 
    u.ID_USUARIO,
    u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
    u.SEMESTRE
FROM USUARIOS u
LEFT JOIN CURSOS_ESTUDIANTES ce ON u.ID_USUARIO = ce.ID_USUARIO
WHERE u.ID_ROL = 1 
  AND ce.ID_USUARIO IS NULL;
'''
run_query(q_d1, "D.1. Estudiantes sin asignaturas matriculadas")


📌 D.1. Estudiantes sin asignaturas matriculadas


,ID_USUARIO,ESTUDIANTE,SEMESTRE
0,12,Cal Stone,1
1,18,Elisabeth Doppler,1


### Consulta D.2: Cursos del catálogo que no tienen horario programado en el calendario
Asignaturas existentes en la malla que no fueron ofertadas en el semestre analizado.


In [26]:
q_d2 = '''
SELECT 
    c.ID_CURSO,
    c.NOMBRE AS CURSO
FROM CURSOS c
LEFT JOIN CALENDARIO_CURSOS cc ON c.ID_CURSO = cc.ID_CURSO
WHERE cc.ID_CURSO IS NULL
ORDER BY c.NOMBRE;
'''
df_d2 = run_query(q_d2, "D.2. Cursos no ofertados en calendario")
print(f"Total de cursos en catálogo sin horario: {len(df_d2)}")
df_d2.head(10)


📌 D.2. Cursos no ofertados en calendario
Total de cursos en catálogo sin horario: 35


,ID_CURSO,CURSO
0,21,Animación y Simulación
1,16,Análisis y Diseño de Algoritmos
2,12,Computabilidad y Complejidad
3,13,Computación Gráfica
4,25,Computación en la Nube
5,14,Comunicación de Datos
6,23,Construcción de Software y Pruebas
7,15,Desarrollo Formal de Sistemas
8,11,Diseño de Interfaces Humano-Computador
9,51,Ecuaciones Diferenciales


### Consulta D.3: Cursos existentes en el sistema sin ningún estudiante matriculado
Cursos que no cuentan con alumnos inscritos en ninguna sección.


In [27]:
q_d3 = '''
SELECT 
    c.ID_CURSO,
    c.NOMBRE AS CURSO
FROM CURSOS c
LEFT JOIN CALENDARIO_CURSOS cc ON c.ID_CURSO = cc.ID_CURSO
LEFT JOIN CURSOS_ESTUDIANTES ce ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
GROUP BY c.ID_CURSO, c.NOMBRE
HAVING COUNT(ce.ID_USUARIO) = 0
ORDER BY c.NOMBRE;
'''
df_d3 = run_query(q_d3, "D.3. Cursos sin estudiantes matriculados")
print(f"Total cursos sin alumnos: {len(df_d3)}")
df_d3.head(10)


📌 D.3. Cursos sin estudiantes matriculados
Total cursos sin alumnos: 48


,ID_CURSO,CURSO
0,21,Animación y Simulación
1,65,Antropología Filosófica
2,16,Análisis y Diseño de Algoritmos
3,20,Aprendizaje Automático
4,12,Computabilidad y Complejidad
5,13,Computación Gráfica
6,25,Computación en la Nube
7,14,Comunicación de Datos
8,23,Construcción de Software y Pruebas
9,15,Desarrollo Formal de Sistemas


### Consulta D.4: Cursos ofertados en calendario pero sin estudiantes matriculados (Cursos desiertos)
Secciones abiertas en programación que tuvieron matrícula cero. Hallazgo crítico para optimización presupuestal y de salones.


In [28]:
q_d4 = '''
SELECT DISTINCT
    cc.ID_CALENDARIO,
    c.ID_CURSO,
    c.NOMBRE AS CURSO,
    cc.DIA,
    cc.HORA_INICIO || ' - ' || cc.HORA_FIN AS HORARIO
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS c ON cc.ID_CURSO = c.ID_CURSO
LEFT JOIN CURSOS_ESTUDIANTES ce ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
WHERE ce.ID_USUARIO IS NULL
ORDER BY c.NOMBRE;
'''
run_query(q_d4, "D.4. Secciones ofertadas con matrícula cero (Cursos desiertos)")


📌 D.4. Secciones ofertadas con matrícula cero (Cursos desiertos)


,ID_CALENDARIO,ID_CURSO,CURSO,DIA,HORARIO
0,33,65,Antropología Filosófica,Martes,08:00 AM - 10:00 AM
1,67,20,Aprendizaje Automático,Martes,04:00 PM - 06:00 PM
2,68,20,Aprendizaje Automático,Jueves,04:00 PM - 06:00 PM
3,7,26,Desarrollo de Videouegos,Martes,02:00 PM - 05:00 PM
4,8,26,Desarrollo de Videouegos,Viernes,02:00 PM - 05:00 PM
5,63,39,Diseño Mecánico,Martes,04:00 PM - 06:00 PM
6,64,39,Diseño Mecánico,Jueves,04:00 PM - 06:00 PM
7,21,52,Estadítica Aplicada,Jueves,06:00 PM - 09:00 PM
8,34,66,Filosofía de la Ciencia,Viernes,06:00 PM - 09:00 PM
9,69,22,Introducción a la Seguridad Informática,Miércoles,04:00 PM - 06:00 PM


---
## Bloque E: Agregaciones, Estadísticas y Cupos (GROUP BY / HAVING)

Métricas institucionales para toma de decisiones: conteos, promedios, demanda de cupos y análisis de capacidad física.


### Consultas E.1 a E.4: Métricas Globales de Estudiantes y Docentes


In [29]:
# E.1: Total de estudiantes
print("--- E.1: Total de estudiantes ---")
display(run_query("SELECT COUNT(*) AS TOTAL_ESTUDIANTES FROM USUARIOS WHERE ID_ROL = 1;"))

# E.2: Total de profesores y total de estudiantes en una sola consulta
print("--- E.2: Distribución de usuarios por rol ---")
q_e2 = '''
SELECT 
    SUM(CASE WHEN ID_ROL = 1 THEN 1 ELSE 0 END) AS TOTAL_ESTUDIANTES,
    SUM(CASE WHEN ID_ROL = 2 THEN 1 ELSE 0 END) AS TOTAL_PROFESORES,
    COUNT(*) AS TOTAL_USUARIOS
FROM USUARIOS;
'''
display(run_query(q_e2))

# E.3: Promedio, mínimo y máximo de semestre
print("--- E.3: Estadísticas del semestre de los estudiantes ---")
q_e3 = '''
SELECT 
    ROUND(AVG(SEMESTRE), 2) AS PROMEDIO_SEMESTRE,
    MIN(SEMESTRE) AS SEMESTRE_MINIMO,
    MAX(SEMESTRE) AS SEMESTRE_MAXIMO
FROM USUARIOS
WHERE ID_ROL = 1;
'''
display(run_query(q_e3))

# E.4: Estudiantes que cursan el semestre más alto
print("--- E.4: Estudiantes en el semestre más avanzado ---")
q_e4 = '''
SELECT 
    ID_USUARIO,
    NOMBRE || ' ' || APELLIDO AS ESTUDIANTE,
    SEMESTRE
FROM USUARIOS
WHERE ID_ROL = 1 
  AND SEMESTRE = (SELECT MAX(SEMESTRE) FROM USUARIOS WHERE ID_ROL = 1);
'''
display(run_query(q_e4))


--- E.1: Total de estudiantes ---


,TOTAL_ESTUDIANTES
0,10


--- E.2: Distribución de usuarios por rol ---


,TOTAL_ESTUDIANTES,TOTAL_PROFESORES,TOTAL_USUARIOS
0,10,10,20


--- E.3: Estadísticas del semestre de los estudiantes ---


,PROMEDIO_SEMESTRE,SEMESTRE_MINIMO,SEMESTRE_MAXIMO
0,4.0,1,6


--- E.4: Estudiantes en el semestre más avanzado ---


,ID_USUARIO,ESTUDIANTE,SEMESTRE
0,19,Magnus Nielsen,6


### Consultas E.5 a E.9: Análisis por Facultades, Carreras y Edificios


In [30]:
# E.5: Cantidad de cursos ofertados por cada facultad
q_e5 = '''
SELECT 
    f.ID_FACULTAD,
    f.NOMBRE AS FACULTAD,
    COUNT(DISTINCT cc.ID_CURSO) AS CURSOS_DISTINTOS,
    COUNT(cc.ID_CALENDARIO) AS TOTAL_SECCIONES_OFERTADAS
FROM FACULTADES f
INNER JOIN CURSOS c ON f.ID_FACULTAD = c.ID_FACULTAD
INNER JOIN CALENDARIO_CURSOS cc ON c.ID_CURSO = cc.ID_CURSO
GROUP BY f.ID_FACULTAD, f.NOMBRE
ORDER BY TOTAL_SECCIONES_OFERTADAS DESC;
'''
display(run_query(q_e5, "E.5. Cursos ofertados por facultad"))

# E.6: Facultades con más de 3 carreras asociadas
q_e6 = '''
SELECT 
    f.ID_FACULTAD,
    f.NOMBRE AS FACULTAD,
    COUNT(c.ID_CARRERA) AS TOTAL_CARRERAS
FROM FACULTADES f
INNER JOIN CARRERAS c ON f.ID_FACULTAD = c.ID_FACULTAD
GROUP BY f.ID_FACULTAD, f.NOMBRE
HAVING COUNT(c.ID_CARRERA) > 3
ORDER BY TOTAL_CARRERAS DESC;
'''
display(run_query(q_e6, "E.6. Facultades con más de 3 programas académicos"))

# E.7 y E.8: Cantidad de estudiantes matriculados por carrera
q_e7 = '''
SELECT 
    car.ID_CARRERA,
    car.NOMBRE AS CARRERA,
    COUNT(ce.ID_USUARIO) AS TOTAL_ESTUDIANTES
FROM CARRERAS car
LEFT JOIN CARRERAS_ESTUDIANTES ce ON car.ID_CARRERA = ce.ID_CARRERA
GROUP BY car.ID_CARRERA, car.NOMBRE
ORDER BY TOTAL_ESTUDIANTES DESC;
'''
display(run_query(q_e7, "E.7. Estudiantes matriculados por carrera"))

# E.9: Edificio con mayor cantidad de salones
q_e9 = '''
SELECT 
    e.ID_EDIFICIO,
    e.NOMBRE AS EDIFICIO,
    COUNT(s.ID_SALON) AS TOTAL_SALONES
FROM EDIFICIOS e
INNER JOIN SALONES s ON e.ID_EDIFICIO = s.ID_EDIFICIO
GROUP BY e.ID_EDIFICIO, e.NOMBRE
ORDER BY TOTAL_SALONES DESC
LIMIT 1;
'''
display(run_query(q_e9, "E.9. Edificio con mayor capacidad de aulas"))


📌 E.5. Cursos ofertados por facultad


,ID_FACULTAD,FACULTAD,CURSOS_DISTINTOS,TOTAL_SECCIONES_OFERTADAS
0,1,Ingeniería y Ciencias,30,65
1,2,Humanidades y Ciencias Sociales,5,6


📌 E.6. Facultades con más de 3 programas académicos


,ID_FACULTAD,FACULTAD,TOTAL_CARRERAS


📌 E.7. Estudiantes matriculados por carrera


,ID_CARRERA,CARRERA,TOTAL_ESTUDIANTES
0,2,Ingeniería Mecánica,4
1,1,Ingeniería de Sistemas,3
2,3,Matemáticas Aplicadas,2
3,4,Filosofía,1


📌 E.9. Edificio con mayor capacidad de aulas


,ID_EDIFICIO,EDIFICIO,TOTAL_SALONES
0,2,El Lago,18


### Consultas E.10 a E.14: Carga Docente, Inscripciones y Análisis de Cupos


In [31]:
# E.10: Total de cursos ofertados por docente
q_e10 = '''
SELECT 
    u.ID_USUARIO,
    u.NOMBRE || ' ' || u.APELLIDO AS PROFESOR,
    COUNT(cc.ID_CALENDARIO) AS TOTAL_SECCIONES,
    COUNT(DISTINCT cc.ID_CURSO) AS CURSOS_DISTINTOS
FROM USUARIOS u
INNER JOIN CALENDARIO_CURSOS cc ON u.ID_USUARIO = cc.ID_PROFESOR
WHERE u.ID_ROL = 2
GROUP BY u.ID_USUARIO, PROFESOR
ORDER BY TOTAL_SECCIONES DESC;
'''
display(run_query(q_e10, "E.10. Carga de cursos por profesor"))

# E.11 y E.12: Matrículas por estudiante y alumnos con menos de 10 asignaturas
q_e12 = '''
SELECT 
    u.ID_USUARIO,
    u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
    u.SEMESTRE,
    COUNT(ce.ID_CALENDARIO) AS TOTAL_CURSOS_INSCRITOS
FROM USUARIOS u
LEFT JOIN CURSOS_ESTUDIANTES ce ON u.ID_USUARIO = ce.ID_USUARIO
WHERE u.ID_ROL = 1
GROUP BY u.ID_USUARIO, ESTUDIANTE, u.SEMESTRE
HAVING COUNT(ce.ID_CALENDARIO) < 10
ORDER BY TOTAL_CURSOS_INSCRITOS ASC;
'''
display(run_query(q_e12, "E.12. Estudiantes con menos de 10 asignaturas inscritas"))

# E.13: Curso con mayor cantidad de matrículas
q_e13 = '''
SELECT 
    c.ID_CURSO,
    c.NOMBRE AS CURSO,
    COUNT(ce.ID_USUARIO) AS TOTAL_INSCRIPCIONES,
    COUNT(DISTINCT ce.ID_USUARIO) AS ESTUDIANTES_UNICOS
FROM CURSOS c
INNER JOIN CALENDARIO_CURSOS cc ON c.ID_CURSO = cc.ID_CURSO
INNER JOIN CURSOS_ESTUDIANTES ce ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
GROUP BY c.ID_CURSO, c.NOMBRE
ORDER BY TOTAL_INSCRIPCIONES DESC
LIMIT 1;
'''
display(run_query(q_e13, "E.13. Asignatura con mayor demanda estudiantil"))

# E.14: Análisis de Demanda y Cupos (secciones con 5 o más inscritos)
q_e14 = '''
SELECT 
    cc.ID_CALENDARIO,
    c.NOMBRE AS CURSO,
    cc.DIA,
    cc.HORA_INICIO || ' - ' || cc.HORA_FIN AS HORARIO,
    cc.CUPO AS CUPO_MAXIMO,
    COUNT(ce.ID_USUARIO) AS ESTUDIANTES_INSCRITOS,
    ROUND((COUNT(ce.ID_USUARIO) * 100.0 / cc.CUPO), 1) AS PORCENTAJE_OCUPACION
FROM CALENDARIO_CURSOS cc
INNER JOIN CURSOS c ON cc.ID_CURSO = c.ID_CURSO
INNER JOIN CURSOS_ESTUDIANTES ce ON cc.ID_CALENDARIO = ce.ID_CALENDARIO
GROUP BY cc.ID_CALENDARIO, c.NOMBRE, cc.DIA, cc.HORA_INICIO, cc.HORA_FIN, cc.CUPO
HAVING COUNT(ce.ID_USUARIO) >= 5
ORDER BY ESTUDIANTES_INSCRITOS DESC;
'''
display(run_query(q_e14, "E.14. Secciones con alta ocupación (>= 5 inscritos)"))


📌 E.10. Carga de cursos por profesor


,ID_USUARIO,PROFESOR,TOTAL_SECCIONES,CURSOS_DISTINTOS
0,6,Saanvi Bahl,18,8
1,1,Ben Stone,8,4
2,2,Grace Stone,8,4
3,4,Jared Vásquez,8,5
4,9,Claudia Tiedemann,8,5
5,3,Mikaela Stone,6,4
6,10,Robert Vance,6,3
7,5,Zeke Landon,4,4
8,7,Ulrich Nielsen,4,2
9,8,Charlotte Doppler,2,1


📌 E.12. Estudiantes con menos de 10 asignaturas inscritas


,ID_USUARIO,ESTUDIANTE,SEMESTRE,TOTAL_CURSOS_INSCRITOS
0,12,Cal Stone,1,0
1,18,Elisabeth Doppler,1,0
2,13,Angelina Meyer,5,2
3,20,Mads Nielsen,2,5


📌 E.13. Asignatura con mayor demanda estudiantil


,ID_CURSO,CURSO,TOTAL_INSCRIPCIONES,ESTUDIANTES_UNICOS
0,28,Cinemática y Dinámica,18,6


📌 E.14. Secciones con alta ocupación (>= 5 inscritos)


,ID_CALENDARIO,CURSO,DIA,HORARIO,CUPO_MAXIMO,ESTUDIANTES_INSCRITOS,PORCENTAJE_OCUPACION
0,13,Cálculo Diferencial,Martes,08:00 AM - 10:00 AM,30,7,23.3
1,14,Cálculo Diferencial,Jueves,08:00 AM - 10:00 AM,30,7,23.3
2,15,Álgebra Lineal,Martes,10:00 AM - 12:00 PM,30,6,20.0
3,16,Álgebra Lineal,Jueves,10:00 AM - 12:00 PM,30,6,20.0
4,23,Cálculo Integral,Martes,10:00 AM - 12:00 PM,20,6,30.0
5,24,Cálculo Integral,Jueves,10:00 AM - 12:00 PM,20,6,30.0
6,25,Cálculo Multivariado,Martes,02:00 PM - 04:00 PM,20,6,30.0
7,26,Cálculo Multivariado,Jueves,02:00 PM - 04:00 PM,20,6,30.0
8,31,Teología,Lunes,10:00 AM - 12:00 PM,30,6,20.0
9,35,Cinemática y Dinámica,Lunes,02:00 PM - 04:00 PM,30,6,20.0


---
# PARTE II: RETOS INTEGRADORES E INTELIGENCIA ARTIFICIAL

---
## Reto Integrador 1: Carga Académica por Carrera (Agregación Múltiple)

**Problema:** Obtener un resumen consolidado por cada carrera que muestre:
* Nombre de la carrera
* Nombre de su facultad
* Total de cursos que componen su malla curricular (CURSOS_CARRERAS)
* Número de estudiantes matriculados en dicha carrera (CARRERAS_ESTUDIANTES)

> ⚠️ **Advertencia Técnica de Analítica:** Si se unen ambas tablas de cardinalidad muchos-a-muchos simultáneamente (INNER JOIN CURSOS_CARRERAS y INNER JOIN CARRERAS_ESTUDIANTES), se genera un **producto cartesiano** que multiplica el conteo de cursos por el número de estudiantes, arrojando valores inflados y erróneos.
>
> **Solución Robusta:** Se utilizan Expresiones de Tabla Comunes (**CTEs** / WITH) para pre-agregar las métricas de forma aislada antes de vincularlas con la tabla principal CARRERAS.


In [32]:
q_reto1 = '''
WITH MallaCurricular AS (
    SELECT 
        ID_CARRERA,
        COUNT(ID_CURSO) AS TOTAL_CURSOS_MALLA
    FROM CURSOS_CARRERAS
    GROUP BY ID_CARRERA
),
MatriculaEstudiantes AS (
    SELECT 
        ID_CARRERA,
        COUNT(ID_USUARIO) AS TOTAL_ESTUDIANTES_MATRICULADOS
    FROM CARRERAS_ESTUDIANTES
    GROUP BY ID_CARRERA
)
SELECT 
    car.ID_CARRERA,
    car.NOMBRE AS CARRERA,
    fac.NOMBRE AS FACULTAD,
    COALESCE(mc.TOTAL_CURSOS_MALLA, 0) AS CURSOS_MALLA,
    COALESCE(me.TOTAL_ESTUDIANTES_MATRICULADOS, 0) AS ESTUDIANTES_MATRICULADOS
FROM CARRERAS car
INNER JOIN FACULTADES fac ON car.ID_FACULTAD = fac.ID_FACULTAD
LEFT JOIN MallaCurricular mc ON car.ID_CARRERA = mc.ID_CARRERA
LEFT JOIN MatriculaEstudiantes me ON car.ID_CARRERA = me.ID_CARRERA
ORDER BY ESTUDIANTES_MATRICULADOS DESC, CURSOS_MALLA DESC;
'''
run_query(q_reto1, "Reto Integrador 1: Consolidado de Malla y Matrícula por Carrera (Solución Anti-Producto Cartesiano)")


📌 Reto Integrador 1: Consolidado de Malla y Matrícula por Carrera (Solución Anti-Producto Cartesiano)


,ID_CARRERA,CARRERA,FACULTAD,CURSOS_MALLA,ESTUDIANTES_MATRICULADOS
0,2,Ingeniería Mecánica,Ingeniería y Ciencias,25,4
1,1,Ingeniería de Sistemas,Ingeniería y Ciencias,37,3
2,3,Matemáticas Aplicadas,Ingeniería y Ciencias,22,2
3,4,Filosofía,Humanidades y Ciencias Sociales,17,1


---
## Reto Integrador 2: Auditoría Text-to-SQL con el Agente de Antigravity (IA / MCP)

### 1. Solicitud en Lenguaje Natural al Agente de IA:
> *\"Identifica qué estudiantes tienen cruces de horario (dos clases el mismo día y a la misma hora) en university_db\".*

---
### 2. Consulta SQL Generada por el Agente de IA y Optimizada:
Para detectar cruces reales sin generar duplicados reflejados ni comparar una clase consigo misma:
1. Se auto-une la tabla CURSOS_ESTUDIANTES usando ce1.ID_USUARIO = ce2.ID_USUARIO y la condición de asimetría ce1.ID_CALENDARIO < ce2.ID_CALENDARIO.
2. Se une con CALENDARIO_CURSOS para validar que coincida el día (c1.DIA = c2.DIA).
3. Se aplica la condición formal de solapamiento de intervalos de tiempo: (c1.HORA_INICIO < c2.HORA_FIN AND c2.HORA_INICIO < c1.HORA_FIN).


In [33]:
q_reto2 = '''
SELECT 
    u.ID_USUARIO,
    u.NOMBRE || ' ' || u.APELLIDO AS ESTUDIANTE,
    c1.DIA,
    cur1.NOMBRE AS CURSO_1,
    c1.HORA_INICIO || ' - ' || c1.HORA_FIN AS HORARIO_1,
    cur2.NOMBRE AS CURSO_2,
    c2.HORA_INICIO || ' - ' || c2.HORA_FIN AS HORARIO_2
FROM CURSOS_ESTUDIANTES ce1
INNER JOIN CURSOS_ESTUDIANTES ce2 
    ON ce1.ID_USUARIO = ce2.ID_USUARIO 
   AND ce1.ID_CALENDARIO < ce2.ID_CALENDARIO
INNER JOIN CALENDARIO_CURSOS c1 ON ce1.ID_CALENDARIO = c1.ID_CALENDARIO
INNER JOIN CALENDARIO_CURSOS c2 ON ce2.ID_CALENDARIO = c2.ID_CALENDARIO
INNER JOIN CURSOS cur1 ON c1.ID_CURSO = cur1.ID_CURSO
INNER JOIN CURSOS cur2 ON c2.ID_CURSO = cur2.ID_CURSO
INNER JOIN USUARIOS u ON ce1.ID_USUARIO = u.ID_USUARIO
WHERE c1.DIA = c2.DIA
  AND (c1.HORA_INICIO < c2.HORA_FIN AND c2.HORA_INICIO < c1.HORA_FIN)
ORDER BY u.ID_USUARIO, c1.DIA, c1.HORA_INICIO;
'''
cruces_df = run_query(q_reto2, "Reto Integrador 2: Cruces de Horario Estudiantiles Auditados")
print(f"Total cruces de horario detectados: {len(cruces_df)}")
cruces_df


📌 Reto Integrador 2: Cruces de Horario Estudiantiles Auditados
Total cruces de horario detectados: 70


,ID_USUARIO,ESTUDIANTE,DIA,CURSO_1,HORARIO_1,CURSO_2,HORARIO_2
0,11,Olive Stone,Jueves,Álgebra Lineal,10:00 AM - 12:00 PM,Cálculo Integral,10:00 AM - 12:00 PM
1,11,Olive Stone,Lunes,Teoría de Probabilidades,08:00 AM - 10:00 AM,Física Térmica y Ondulatoria,08:00 AM - 10:00 AM
2,11,Olive Stone,Martes,Álgebra Lineal,10:00 AM - 12:00 PM,Cálculo Integral,10:00 AM - 12:00 PM
3,11,Olive Stone,Miércoles,Teoría de Probabilidades,08:00 AM - 10:00 AM,Física Térmica y Ondulatoria,08:00 AM - 10:00 AM
4,14,Jonas Kahnwald,Jueves,Cálculo Multivariado,02:00 PM - 04:00 PM,Propiedades de los Materiales,02:00 PM - 04:00 PM
...,...,...,...,...,...,...,...
65,19,Magnus Nielsen,Martes,Mecánica de Sólidos,09:00 AM - 12:00 PM,Química y Ciencia de Materiales,08:00 AM - 10:00 AM
66,19,Magnus Nielsen,Martes,Álgebra Lineal,10:00 AM - 12:00 PM,Cálculo Integral,10:00 AM - 12:00 PM
67,19,Magnus Nielsen,Martes,Álgebra Lineal,10:00 AM - 12:00 PM,Mecánica de Sólidos,09:00 AM - 12:00 PM
68,19,Magnus Nielsen,Martes,Cálculo Integral,10:00 AM - 12:00 PM,Mecánica de Sólidos,09:00 AM - 12:00 PM


### 3. Análisis Crítico y Auditoría de la Generación Text-to-SQL

1. **Riesgo de Coincidencia Trivial (Misma sección):**
   * *Error potencial de agentes de IA:* Un agente básico suele escribir ce1.ID_CALENDARIO != ce2.ID_CALENDARIO. Si bien evita comparar una fila consigo misma, reporta el mismo conflicto dos veces: (Clase A vs Clase B) y (Clase B vs Clase A).
   * *Corrección aplicada:* Utilizar el operador relacional de orden estricto ce1.ID_CALENDARIO < ce2.ID_CALENDARIO. Esto garantiza que cada par de conflicto aparezca exactamente una sola vez.

2. **Riesgo en el Solapamiento Temporal Parcial:**
   * *Error potencial de agentes de IA:* Comparar igualdad exacta de horas de inicio: c1.HORA_INICIO = c2.HORA_INICIO.
   * *Fallo analítico:* Si una clase inicia a las 08:00 y termina a las 10:00, y otra inicia a las 09:00 y termina a las 11:00, hay un choque real de 1 hora que no se detectaría con simple igualdad.
   * *Corrección aplicada:* Fórmula canónica de traslape de rangos continuos: c1.HORA_INICIO < c2.HORA_FIN AND c2.HORA_INICIO < c1.HORA_FIN. Al almacenar las horas en formato HH:MM:SS (ISO-24H), la comparación de cadenas léxicas en SQLite es matemáticamente equivalente a la comparación cronológica.

3. **Relevancia para la Toma de Decisiones Institucional:**
   * La detección automatizada de cruces permite al sistema de registro bloquear inscripciones simultáneas incompatibles en tiempo real (*Trigger* de base de datos) o notificar a los estudiantes afectados para reajuste de horario antes del cierre de adiciones.


---
## 4. Conclusiones y Cierre del Taller

1. **Consistencia Relacional y Normalización:**
   * La estructura de 11 tablas de university.db separa adecuadamente la infraestructura física (EDIFICIOS, SALONES), la oferta académica (CURSOS, FACULTADES, CARRERAS) y la dinámica operacional de eventos (CALENDARIO_CURSOS, CURSOS_ESTUDIANTES).

2. **Auditoría de Registros Huérfanos:**
   * El uso de LEFT JOIN ... WHERE IS NULL demostró ser indispensable para la gobernanza de datos, revelando asignaturas del catálogo que nunca fueron programadas y secciones abiertas con cupos asignados que no captaron matrícula.

3. **Prevención de Errores de Agregación:**
   * En analítica avanzada, cruzar mallas curriculares y matrículas exige aislar los niveles de granularidad mediante subconsultas o CTEs para evitar distorsiones por productos cartesianos.

4. **Sinergia con Agentes Text-to-SQL:**
   * La inteligencia artificial acelera la redacción de consultas complejas, pero el rol del científico de datos como auditor crítico es insustituible para validar asimetrías de llaves y comparaciones de rangos temporales.


In [34]:
# Cierre formal de la conexión a la base de datos
conn.close()
print("🔒 Conexión a university.db cerrada correctamente. Taller finalizado.")


🔒 Conexión a university.db cerrada correctamente. Taller finalizado.
